In [ ]:
from pathlib import Path
from pprint import pprint

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
PROJECT_ROOT = Path().resolve().parent
REPORT_PATH = PROJECT_ROOT / "report-2026-06-19-interscada-pl.joblib"

data = joblib.load(REPORT_PATH)
len(data)

47943

In [ ]:
pprint(data[0])

{'cct_true': 0.5,
 'cct_weighted_global': 0.4999999999999998,
 'cct_weighted_per_location': 0.49999999999999994,
 'crit_gen_true': '<na>',
 'distance_mean': 2.5360972885146356,
 'distance_median': 2.6238584642020055,
 'distance_min': 0.7927440682961193,
 'distance_norm': 0.3021291274325327,
 'distance_spread': 2.692970251751999,
 'has_crit_gen_prediction': True,
 'has_location_prediction': True,
 'location_neighbor_count': 61,
 'location_true': 'bus1',
 'location_weight_mass': 0.12635561438194176,
 'n_eff': 300.64715340304406,
 'n_neighbors': 513,
 'neighborhood_compactness': 0.10031753873770247,
 'prediction_summary': ReportSummary(cct_weighted=0.4999999999999998, cct_weighted_per_location={'BUS1': 0.49999999999999994, 'BUS12': 0.5, 'BUS28': 0.5000000000000001, 'BUS9': 0.5, '_3f_50_BUS1 -BUS2': 0.49999999999999994, '_3f_50_BUS1 -BUS40': 0.5, '_3f_50_BUS25-BUS26': 0.5, '_3f_50_BUS26-BUS27': 0.5, '_3f_50_BUS28-BUS29': 0.5, '_3f_50_BUS41-BUS28': 0.5, '_3f_50_BUS41-BUS29': 0.4999999999999

In [ ]:
def summary_value(summary, name: str, default=None):
    return getattr(summary, name, default) if summary is not None else default


df = pd.DataFrame(
    {
        "state": str(d["state"]),
        "cct_true": float(d["cct_true"]),
        "crit_gen_true": d["crit_gen_true"],
        "location_true": d["location_true"],
        "cct_weighted_per_location": d.get("cct_weighted_per_location"),
        "cct_weighted_global": d.get("cct_weighted_global", summary_value(d.get("prediction_summary"), "cct_weighted")),
        "has_crit_gen_prediction": d.get("has_crit_gen_prediction", d.get("prediction_summary") is not None),
        "has_location_prediction": d.get("has_location_prediction", d.get("cct_weighted_per_location") is not None),
        "location_weight_mass": d.get("location_weight_mass"),
        "location_neighbor_count": d.get("location_neighbor_count"),
        "n_neighbors": d.get("n_neighbors", summary_value(d.get("prediction_summary"), "n")),
        "n_eff": d.get("n_eff", summary_value(d.get("prediction_summary"), "n_eff")),
        "neighborhood_compactness": d.get("neighborhood_compactness"),
        "distance_min": d.get("distance_min"),
        "distance_mean": d.get("distance_mean"),
        "distance_median": d.get("distance_median"),
        "distance_spread": d.get("distance_spread"),
        "distance_norm": d.get("distance_norm"),
    }
    for d in data
)

df.head()

,state,cct_true,crit_gen_true,location_true,cct_weighted_per_location,cct_weighted_global,has_crit_gen_prediction,has_location_prediction,location_weight_mass,location_neighbor_count,n_neighbors,n_eff,neighborhood_compactness,distance_min,distance_mean,distance_median,distance_spread,distance_norm
0,SET_0001__Point_no.___11,0.500,<na>,bus1,0.500000,0.500000,True,True,0.126356,61,513,300.647153,0.100318,0.792744,2.536097,2.623858,2.692970,0.302129
1,SET_0001__Point_no.___11,0.249,30,bus2,0.265458,0.300696,True,True,0.019569,12,484,281.159255,0.117308,0.792744,2.401742,2.523040,2.692970,0.314202
2,SET_0001__Point_no.___11,0.238,31,bus3,0.227113,0.270954,True,True,0.029214,42,1498,958.020539,0.099950,0.792744,2.473086,2.557373,2.692970,0.309984
3,SET_0001__Point_no.___11,0.244,36,bus4,0.218309,0.239352,True,True,0.070838,12,274,172.804745,0.114918,0.792744,2.459825,2.557373,2.661412,0.309984
4,SET_0001__Point_no.___11,0.230,30,bus5,0.238472,0.300696,True,True,0.005358,4,484,281.159255,0.117308,0.792744,2.401742,2.523040,2.692970,0.314202


In [ ]:
coverage = pd.Series(
    {
        "n_total": len(df),
        "n_with_crit_gen_prediction": int(df["has_crit_gen_prediction"].sum()),
        "n_with_location_prediction": int(df["has_location_prediction"].sum()),
        "crit_gen_coverage": float(df["has_crit_gen_prediction"].mean()),
        "location_coverage": float(df["has_location_prediction"].mean()),
        "n_missing_crit_gen_prediction": int((~df["has_crit_gen_prediction"]).sum()),
        "n_missing_location_prediction": int((~df["has_location_prediction"]).sum()),
    }
)

coverage

n_total                          47943.000000
n_with_crit_gen_prediction       47943.000000
n_with_location_prediction       47661.000000
crit_gen_coverage                    1.000000
location_coverage                    0.994118
n_missing_crit_gen_prediction        0.000000
n_missing_location_prediction      282.000000
dtype: float64

In [ ]:
missing = df.loc[~df["has_location_prediction"]].copy()
missing_by_crit_gen = missing["crit_gen_true"].value_counts()
missing_by_location = missing["location_true"].value_counts().head(20)

display(missing_by_crit_gen)
display(missing_by_location)

crit_gen_true
39      44
35      39
32      35
30      31
33      28
38      28
36      25
37      24
31      15
<na>    11
34       2
Name: count, dtype: int64

location_true
_3f_50_bus3 -bus18    8
bus12                 8
bus21                 8
_3f_50_bus2 -bus25    7
bus16                 7
_3f_50_bus26-bus27    6
bus26                 6
_3f_50_bus2 -bus3     6
_3f_50_bus22-bus23    6
_3f_50_bus28-bus29    6
bus22                 6
_3f_50_bus14-bus15    6
bus25                 6
_3f_50_bus17-bus18    6
bus3                  6
_3f_50_bus16-bus21    6
_3f_50_bus21-bus22    6
_3f_50_bus8 -bus9     6
bus9                  6
bus29                 5
Name: count, dtype: int64

In [ ]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> pd.Series:
    valid = frame[pred_col].notna()
    n_valid = int(valid.sum())

    if n_valid == 0:
        return pd.Series(
            {
                "n": 0,
                "coverage": 0.0,
                "mse": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "abs_err_min": np.nan,
                "abs_err_q25": np.nan,
                "abs_err_q50": np.nan,
                "abs_err_q75": np.nan,
                "abs_err_q90": np.nan,
                "abs_err_q95": np.nan,
                "abs_err_q99": np.nan,
                "abs_err_max": np.nan,
            }
        )

    y_true = frame.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = frame.loc[valid, pred_col].to_numpy(dtype=float)
    err = np.abs(y_pred - y_true)

    return pd.Series(
        {
            "n": n_valid,
            "coverage": float(valid.mean()),
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "abs_err_min": np.quantile(err, 0.00),
            "abs_err_q25": np.quantile(err, 0.25),
            "abs_err_q50": np.quantile(err, 0.50),
            "abs_err_q75": np.quantile(err, 0.75),
            "abs_err_q90": np.quantile(err, 0.90),
            "abs_err_q95": np.quantile(err, 0.95),
            "abs_err_q99": np.quantile(err, 0.99),
            "abs_err_max": np.quantile(err, 1.00),
        }
    )


metrics_conditional = pd.DataFrame(
    {
        "per_location": regression_metrics(df, "cct_weighted_per_location"),
        "global_fallback": regression_metrics(df, "cct_weighted_global"),
    }
)

df["cct_pred_overall"] = df["cct_weighted_per_location"].fillna(df["cct_weighted_global"])
metrics_overall = pd.DataFrame(
    {
        "location_then_global": regression_metrics(df, "cct_pred_overall"),
    }
)

metrics_conditional, metrics_overall

(             per_location  global_fallback
 n            47661.000000     47943.000000
 coverage         0.994118         1.000000
 mse              0.000789         0.005129
 rmse             0.028083         0.071614
 mae              0.017741         0.051119
 abs_err_min      0.000000         0.000000
 abs_err_q25      0.003000         0.013330
 abs_err_q50      0.011140         0.038852
 abs_err_q75      0.024881         0.073202
 abs_err_q90      0.042647         0.114860
 abs_err_q95      0.057565         0.151731
 abs_err_q99      0.097906         0.232758
 abs_err_max      0.415111         0.360407,
              location_then_global
 n                    47943.000000
 coverage                 1.000000
 mse                      0.000858
 rmse                     0.029285
 mae                      0.018139
 abs_err_min              0.000000
 abs_err_q25              0.003032
 abs_err_q50              0.011236
 abs_err_q75              0.025143
 abs_err_q90              0.04327

In [ ]:
by_crit_gen = (
    df.groupby("crit_gen_true", observed=True)
    .apply(lambda g: regression_metrics(g, "cct_weighted_per_location"), include_groups=False)
    .sort_values("mae")
    .astype({"n": int})
)

by_crit_gen.to_csv("./interscada_pl_report.csv")

by_crit_gen

,n,coverage,mse,rmse,mae,abs_err_min,abs_err_q25,abs_err_q50,abs_err_q75,abs_err_q90,abs_err_q95,abs_err_q99,abs_err_max
crit_gen_true,,,,,,,,,,,,,
<na>,5499,0.998004,6.015598e-33,7.756029e-17,5.359314e-17,0.0,0.000000,5.551115e-17,1.110223e-16,1.110223e-16,1.110223e-16,2.220446e-16,3.330669e-16
38,2471,0.988796,5.969058e-04,2.443166e-02,1.641523e-02,0.0,0.005459,1.179750e-02,2.096165e-02,3.500993e-02,4.654780e-02,9.259268e-02,1.898031e-01
31,8884,0.998314,7.042208e-04,2.653716e-02,1.731326e-02,0.0,0.004082,1.059748e-02,2.457383e-02,4.014594e-02,5.300891e-02,8.838498e-02,3.075806e-01
37,5837,0.995905,8.721679e-04,2.953249e-02,2.008516e-02,0.0,0.005324,1.323660e-02,2.776574e-02,4.582329e-02,6.027680e-02,1.041633e-01,2.235029e-01
32,2565,0.986538,1.004269e-03,3.169021e-02,2.102354e-02,0.0,0.004899,1.400000e-02,2.931326e-02,4.851939e-02,6.403321e-02,1.012651e-01,2.448162e-01
30,1775,0.982835,9.599777e-04,3.098351e-02,2.106731e-02,0.0,0.007247,1.566223e-02,2.795603e-02,4.438923e-02,5.807869e-02,9.097451e-02,4.151110e-01
35,3106,0.987599,1.086463e-03,3.296153e-02,2.113660e-02,0.0,0.005910,1.396740e-02,2.721666e-02,4.701189e-02,6.578681e-02,1.324270e-01,3.138988e-01
36,2650,0.990654,1.056120e-03,3.249799e-02,2.135633e-02,0.0,0.006033,1.498134e-02,2.690129e-02,4.599707e-02,6.584402e-02,1.189761e-01,3.200983e-01
39,390,0.898618,1.382222e-03,3.717825e-02,2.135771e-02,0.0,0.005811,1.434508e-02,2.794072e-02,4.239904e-02,5.662310e-02,1.588661e-01,2.950000e-01


In [ ]:
missing_diagnostics = {
    "missing_by_crit_gen": df.loc[~df["has_crit_gen_prediction"], "crit_gen_true"].value_counts().head(20),
    "missing_by_location": df.loc[~df["has_location_prediction"], "location_true"].value_counts().head(20),
}

missing_diagnostics

{'missing_by_crit_gen': Series([], Name: count, dtype: int64),
 'missing_by_location': location_true
 _3f_50_bus3 -bus18    8
 bus12                 8
 bus21                 8
 _3f_50_bus2 -bus25    7
 bus16                 7
 _3f_50_bus26-bus27    6
 bus26                 6
 _3f_50_bus2 -bus3     6
 _3f_50_bus22-bus23    6
 _3f_50_bus28-bus29    6
 bus22                 6
 _3f_50_bus14-bus15    6
 bus25                 6
 _3f_50_bus17-bus18    6
 bus3                  6
 _3f_50_bus16-bus21    6
 _3f_50_bus21-bus22    6
 _3f_50_bus8 -bus9     6
 bus9                  6
 bus29                 5
 Name: count, dtype: int64}

In [ ]:
from scipy.stats import spearmanr


def analysis():
    _df = pd.DataFrame(data)
    _df = _df.drop(columns=["prediction_summary"])
    _df = _df.dropna(subset=["cct_weighted_per_location"])
    _df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

    rho, p_value = spearmanr(_df.err, _df.n_eff)
    print("n_eff", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.neighborhood_compactness)
    print("neighborhood_compactness", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.n_neighbors)
    print("n_neighbors", f"{rho=} {p_value=}")

    rho, p_value = spearmanr(_df.err, _df.location_weight_mass)
    print("location_weight_mass", f"{rho=} {p_value=}")

    dist_cols = [c for c in _df.columns if c.startswith("distance")]
    for col in dist_cols:
        rho, p_value = spearmanr(_df.err, _df[col])
        print(col, f"{rho=} {p_value=}")


analysis()

n_eff rho=np.float64(-0.023609549312327567) p_value=np.float64(2.537957178687328e-07)
neighborhood_compactness rho=np.float64(0.04127095632826745) p_value=np.float64(1.9934434292105247e-19)
n_neighbors rho=np.float64(-0.07530152377360438) p_value=np.float64(6.842600545357078e-61)
location_weight_mass rho=np.float64(-0.15477304324189833) p_value=np.float64(2.802922736676538e-253)
distance_min rho=np.float64(0.03235457430591991) p_value=np.float64(1.6047652506526248e-12)
distance_mean rho=np.float64(-0.07759204301997863) p_value=np.float64(1.5021180148504688e-64)
distance_median rho=np.float64(-0.07013034575694702) p_value=np.float64(4.901469635885509e-53)
distance_spread rho=np.float64(-0.12238160749785527) p_value=np.float64(2.0051501461036158e-158)
distance_norm rho=np.float64(0.16412443378114924) p_value=np.float64(5.6775375890242654e-285)


In [ ]:
_df = pd.DataFrame(data)
_df = _df.drop(columns=["prediction_summary"])
_df = _df.dropna(subset=["cct_weighted_per_location"])
_df["err"] = (_df["cct_true"] - _df["cct_weighted_per_location"]).abs()

In [ ]:
def risk_coverage(df, metric, higher_is_better=True, coverages=(1.0, 0.95, 0.9, 0.8, 0.7, 0.5)):
    x = df.dropna(subset=[metric, "err"]).copy()
    x = x.sort_values(metric, ascending=not higher_is_better)

    rows = []
    n = len(x)
    for cov in coverages:
        k = int(np.ceil(cov * n))
        kept = x.iloc[:k]
        rows.append(
            {
                "metric": metric,
                "coverage": cov,
                "n": k,
                "mae": kept["err"].mean(),
                "rmse": np.sqrt((kept["err"] ** 2).mean()),
                "q90": kept["err"].quantile(0.90),
                "q95": kept["err"].quantile(0.95),
            }
        )
    return pd.DataFrame(rows)


metrics = {
    "location_weight_mass": True,
    "n_eff": True,
    "n_neighbors": True,
    "neighborhood_compactness": True,
    "distance_min": False,
    "distance_mean": False,
    "distance_median": False,
    "distance_spread": False,
    "distance_norm": False,
}

out = []
for metric, higher_is_better in metrics.items():
    out.append(risk_coverage(_df, metric, higher_is_better))

rc = pd.concat(out, ignore_index=True)
rc.to_csv("./risk_coverage_interscada_pl.csv")
rc

,metric,coverage,n,mae,rmse,q90,q95
0,location_weight_mass,1.00,47661,0.017741,0.028083,0.042647,0.057565
1,location_weight_mass,0.95,45278,0.016948,0.026285,0.040952,0.054880
2,location_weight_mass,0.90,42895,0.016567,0.025749,0.040208,0.053794
3,location_weight_mass,0.80,38129,0.016232,0.025578,0.039943,0.053575
4,location_weight_mass,0.70,33363,0.016370,0.026054,0.040723,0.054900
5,location_weight_mass,0.50,23831,0.016518,0.026488,0.040841,0.055382
6,n_eff,1.00,47661,0.017741,0.028083,0.042647,0.057565
7,n_eff,0.95,45278,0.017842,0.027669,0.042827,0.057477
8,n_eff,0.90,42895,0.018054,0.027719,0.043274,0.057862
9,n_eff,0.80,38129,0.018109,0.027684,0.043448,0.058050
